In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

Mounted at /content/drive


In [3]:
BASE_DIR = "/content/drive/MyDrive/thesis_movie_project"
RAW_DIR = f"{BASE_DIR}/raw"
PROCESSED_DIR = f"{BASE_DIR}/processed"

RAW_IMDB_FILENAME = "TMDB_all_movies.csv"
RAW_IMDB_PATH = f"{RAW_DIR}/{RAW_IMDB_FILENAME}"

LOCAL_RAW_DIR = "/content/thesis_movie_project/raw"
LOCAL_IMDB_PATH = f"{LOCAL_RAW_DIR}/{RAW_IMDB_FILENAME}"

PRIMARY_OUT = f"{PROCESSED_DIR}/imdb_master_primary.csv"
AUDIT_OUT = f"{PROCESSED_DIR}/imdb_master_primary_audit.csv"
DEDUP_LOG_OUT = f"{PROCESSED_DIR}/imdb_master_primary_dedup_log.csv"
YEAR_COUNT_OUT = f"{PROCESSED_DIR}/imdb_master_primary_movies_per_year.csv"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

print("RAW_IMDB_PATH:", RAW_IMDB_PATH)
print("PRIMARY_OUT:", PRIMARY_OUT)

RAW_IMDB_PATH: /content/drive/MyDrive/thesis_movie_project/raw/TMDB_all_movies.csv
PRIMARY_OUT: /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary.csv


In [4]:
if not os.path.exists(LOCAL_IMDB_PATH):
    print("Copying raw IMDb file to local Colab storage...")
    shutil.copy2(RAW_IMDB_PATH, LOCAL_IMDB_PATH)
else:
    print("Local copy already exists.")

print("Local file ready:", LOCAL_IMDB_PATH)

Copying raw IMDb file to local Colab storage...
Local file ready: /content/thesis_movie_project/raw/TMDB_all_movies.csv


In [5]:
sample = pd.read_csv(LOCAL_IMDB_PATH, low_memory=False, nrows=5)

print("Raw columns:")
for c in sample.columns:
    print("-", c)

display(sample.head())

Raw columns:
- id
- title
- vote_average
- vote_count
- status
- release_date
- revenue
- runtime
- budget
- imdb_id
- original_language
- original_title
- overview
- popularity
- tagline
- genres
- production_companies
- production_countries
- spoken_languages
- cast
- director
- director_of_photography
- writers
- producers
- music_composer
- imdb_rating
- imdb_votes
- poster_path


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9758.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, English","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8639.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.900,2821.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,English,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Andrzej Sekula, Rodrigo García, Phil Parmet, G...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Alexandre Rockwell, Lawrenc...",Combustible Edison,6.7,116925.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.500,370.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,English,"Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Lloyd Segan, Gene Levy, Marilyn Vance",Alan Silvestri,6.6,21069.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.200,30.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


In [6]:
def clean_text_col(series):
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def empty_to_nan(series):
    s = clean_text_col(series)
    return s.replace({
        "": np.nan,
        "Unknown": np.nan,
        "unknown": np.nan,
        "UNKNOWN": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "None": np.nan,
        "NONE": np.nan,
        "N/A": np.nan,
        "n/a": np.nan,
        "null": np.nan,
        "NULL": np.nan,
        "[]": np.nan
    })

def to_num(series):
    return pd.to_numeric(series, errors="coerce")

def pick_col(available_cols, candidates):
    for c in candidates:
        if c in available_cols:
            return c
    return None

def parse_listlike_fast(x, top_n=None):
    """
    Fast parser for strings like:
    "['Action', 'Drama']"
    "['Tom Hanks', 'Meg Ryan']"
    Falls back to light regex extraction.
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        vals = [str(v).strip() for v in x if str(v).strip()]
        return vals[:top_n] if top_n else vals

    s = str(x).strip()
    if s == "" or s == "[]" or s.lower() == "nan":
        return []

    # extract quoted items
    vals = re.findall(r"'([^']+)'|\"([^\"]+)\"", s)
    flat = []
    for a, b in vals:
        v = a if a else b
        v = str(v).strip()
        if v:
            flat.append(v)

    if not flat:
        # fallback: split naive comma if no quotes found
        s2 = s.strip("[]")
        parts = [p.strip(" '\"") for p in s2.split(",")]
        flat = [p for p in parts if p]

    return flat[:top_n] if top_n else flat

def list_to_string(series, top_n=None):
    return series.apply(lambda x: " ".join(parse_listlike_fast(x, top_n=top_n)).strip())

def normalize_title(series):
    s = clean_text_col(series).str.lower()
    s = s.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

In [7]:
raw_head = pd.read_csv(LOCAL_IMDB_PATH, low_memory=False, nrows=1)
available_cols = list(raw_head.columns)

col_id = pick_col(available_cols, ["id"])
col_title = pick_col(available_cols, ["title"])
col_imdb_id = pick_col(available_cols, ["imdb_id"])
col_overview = pick_col(available_cols, ["overview"])
col_release_date = pick_col(available_cols, ["release_date"])
col_release_year = pick_col(available_cols, ["release_year"])
col_status = pick_col(available_cols, ["status"])
col_budget = pick_col(available_cols, ["budget"])
col_revenue = pick_col(available_cols, ["revenue"])
col_runtime = pick_col(available_cols, ["runtime"])
col_popularity = pick_col(available_cols, ["popularity"])
col_vote_average = pick_col(available_cols, ["vote_average"])
col_vote_count = pick_col(available_cols, ["vote_count"])
col_original_language = pick_col(available_cols, ["original_language"])

col_keywords = pick_col(available_cols, ["keywords", "all_combined_keywords"])
col_keywords_fallback = pick_col(available_cols, ["all_combined_keywords"])
col_genres = pick_col(available_cols, ["genres_list", "genres"])
col_cast = pick_col(available_cols, ["Cast_list", "cast"])
col_director = pick_col(available_cols, ["Director", "director_name"])

usecols = [
    c for c in [
        col_id, col_title, col_imdb_id, col_overview, col_release_date, col_release_year,
        col_status, col_budget, col_revenue, col_runtime, col_popularity, col_vote_average,
        col_vote_count, col_original_language, col_keywords, col_keywords_fallback,
        col_genres, col_cast, col_director
    ]
    if c is not None
]

usecols = sorted(set(usecols))

print("Selected columns:")
for c in usecols:
    print("-", c)

Selected columns:
- budget
- cast
- genres
- id
- imdb_id
- original_language
- overview
- popularity
- release_date
- revenue
- runtime
- status
- title
- vote_average
- vote_count


In [8]:
raw = pd.read_csv(
    LOCAL_IMDB_PATH,
    usecols=usecols,
    low_memory=False
)

print("Loaded raw subset shape:", raw.shape)
display(raw.head(3))

Loaded raw subset shape: (1192309, 15)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,overview,popularity,genres,cast
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,fi,A Finnish man goes to the city to find a job a...,1.6384,"Comedy, Drama, Romance, Crime","Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,..."
1,3,Shadows in Paradise,7.300,435.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,fi,"Nikander, a rubbish collector and would-be ent...",2.7136,"Comedy, Drama, Romance","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ..."
2,5,Four Rooms,5.900,2821.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,en,It's Ted the Bellhop's first night on the job....,4.0661,Comedy,"David Proval, Tamlyn Tomita, Paul Skemp, Lana ..."


In [9]:
master = pd.DataFrame()

master["id"] = to_num(raw[col_id]) if col_id else np.nan
master["title"] = clean_text_col(raw[col_title]) if col_title else ""
master["imdb_id"] = clean_text_col(raw[col_imdb_id]) if col_imdb_id else ""
master["overview"] = clean_text_col(raw[col_overview]) if col_overview else ""

master["release_date"] = pd.to_datetime(raw[col_release_date], errors="coerce") if col_release_date else pd.NaT

if col_release_year:
    master["release_year"] = to_num(raw[col_release_year])
else:
    master["release_year"] = master["release_date"].dt.year

master["status"] = clean_text_col(raw[col_status]) if col_status else ""

master["budget"] = to_num(raw[col_budget]) if col_budget else np.nan
master["revenue"] = to_num(raw[col_revenue]) if col_revenue else np.nan
master["runtime"] = to_num(raw[col_runtime]) if col_runtime else np.nan
master["popularity"] = to_num(raw[col_popularity]) if col_popularity else np.nan
master["vote_average"] = to_num(raw[col_vote_average]) if col_vote_average else np.nan
master["vote_count"] = to_num(raw[col_vote_count]) if col_vote_count else np.nan
master["original_language"] = clean_text_col(raw[col_original_language]) if col_original_language else ""

# semantic helpers
if col_keywords:
    master["keywords_clean"] = list_to_string(raw[col_keywords], top_n=20)
else:
    master["keywords_clean"] = ""

if col_keywords_fallback and col_keywords_fallback != col_keywords:
    kw_fb = list_to_string(raw[col_keywords_fallback], top_n=20)
    master["keywords_clean"] = np.where(
        clean_text_col(master["keywords_clean"]).str.len() > 0,
        master["keywords_clean"],
        kw_fb
    )

master["genres_clean"] = list_to_string(raw[col_genres], top_n=None) if col_genres else ""
master["cast_clean"] = list_to_string(raw[col_cast], top_n=5) if col_cast else ""
master["director_name"] = clean_text_col(raw[col_director]) if col_director else ""

master["movieId"] = pd.Series([pd.NA] * len(master), dtype="Int64")

# placeholder normalization
for col in ["title", "imdb_id", "overview", "keywords_clean", "genres_clean", "cast_clean", "director_name", "original_language", "status"]:
    master[col] = empty_to_nan(master[col]).fillna("")

# cleaned finance
master["budget_clean"] = to_num(master["budget"])
master["revenue_clean"] = to_num(master["revenue"])

master.loc[master["budget_clean"] <= 0, "budget_clean"] = np.nan
master.loc[master["revenue_clean"] <= 0, "revenue_clean"] = np.nan

master["roi_clean"] = np.where(
    master["budget_clean"].notna() & master["revenue_clean"].notna(),
    master["revenue_clean"] / master["budget_clean"],
    np.nan
)
master["roi_raw"] = master["roi_clean"]

# text fields
master["overview_len"] = clean_text_col(master["overview"]).str.len()
master["overview_word_count"] = clean_text_col(master["overview"]).str.split().str.len()

master["text_overview_only"] = clean_text_col(master["overview"])
master["text_semantic_enriched"] = (
    clean_text_col(master["title"]) + " " +
    clean_text_col(master["keywords_clean"]) + " " +
    clean_text_col(master["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

master["text_semantic_full"] = (
    clean_text_col(master["title"]) + " " +
    clean_text_col(master["keywords_clean"]) + " " +
    clean_text_col(master["cast_clean"]) + " " +
    clean_text_col(master["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

# provenance
master["source_name"] = RAW_IMDB_FILENAME

print("Standardized master shape:", master.shape)
display(master.head(3))

/tmp/ipykernel_5195/3499445174.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return s.replace({
/tmp/ipykernel_5195/3499445174.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return s.replace({


Standardized master shape: (1192309, 29)


,id,title,imdb_id,overview,release_date,release_year,status,budget,revenue,runtime,...,budget_clean,revenue_clean,roi_clean,roi_raw,overview_len,overview_word_count,text_overview_only,text_semantic_enriched,text_semantic_full,source_name
0,2,Ariel,tt0094675,A Finnish man goes to the city to find a job a...,1988-10-21,1988.0,Released,0.0,0.0,73.0,...,NaN,NaN,NaN,NaN,117,24,A Finnish man goes to the city to find a job a...,Ariel A Finnish man goes to the city to find a...,Ariel Kari Helaseppä Jaakko Talaskivi Mikko Re...,TMDB_all_movies.csv
1,3,Shadows in Paradise,tt0092149,"Nikander, a rubbish collector and would-be ent...",1986-10-17,1986.0,Released,0.0,0.0,74.0,...,NaN,NaN,NaN,NaN,258,38,"Nikander, a rubbish collector and would-be ent...","Shadows in Paradise Nikander, a rubbish collec...",Shadows in Paradise Ari Korhonen Mari Rantasil...,TMDB_all_movies.csv
2,5,Four Rooms,tt0113101,It's Ted the Bellhop's first night on the job....,1995-12-09,1995.0,Released,4000000.0,4257354.0,98.0,...,4000000.0,4257354.0,1.064339,1.064339,237,38,It's Ted the Bellhop's first night on the job....,Four Rooms It's Ted the Bellhop's first night ...,Four Rooms David Proval Tamlyn Tomita Paul Ske...,TMDB_all_movies.csv


In [10]:
today = pd.Timestamp.today().normalize()

master["is_released_row"] = (
    (master["status"] == "") |
    (master["status"].str.lower() == "released")
)

master["is_future_release"] = (
    master["release_date"].notna() &
    (master["release_date"] > today)
)

master["has_valid_year"] = master["release_year"].between(1888, 2026, inclusive="both")
master["has_usable_title"] = clean_text_col(master["title"]).str.len() > 0
master["has_usable_overview"] = master["overview_word_count"].fillna(0) >= 10

master["has_valid_runtime"] = master["runtime"].between(40, 300, inclusive="both")
master["is_feature_length"] = master["runtime"].fillna(0) >= 60
master["is_short_runtime"] = master["runtime"].fillna(0) < 60

master["has_genres"] = clean_text_col(master["genres_clean"]).str.len() > 0
master["has_cast"] = clean_text_col(master["cast_clean"]).str.len() > 0
master["has_director"] = clean_text_col(master["director_name"]).str.len() > 0
master["has_imdb_id"] = clean_text_col(master["imdb_id"]).str.len() > 0

master["has_finance_signal"] = master["revenue_clean"].notna() | master["budget_clean"].notna()
master["has_audience_signal"] = (
    (master["vote_count"].fillna(0) > 0) |
    (master["popularity"].fillna(0) > 0)
)

master["is_tv_movie"] = master["genres_clean"].fillna("").str.contains(r"\bTV Movie\b", regex=True)
master["has_valid_language"] = clean_text_col(master["original_language"]).str.len() > 0

display(
    master[[
        "is_released_row", "is_future_release", "has_valid_year", "has_usable_title",
        "has_usable_overview", "has_valid_runtime", "is_feature_length", "has_genres",
        "has_cast", "has_director", "has_imdb_id", "has_finance_signal", "has_audience_signal",
        "is_tv_movie"
    ]].mean().sort_values(ascending=False).round(4) * 100
)

,0
has_usable_title,100.00
has_audience_signal,99.65
is_released_row,98.17
has_valid_year,89.43
has_usable_overview,77.83
has_genres,73.07
has_cast,68.24
has_imdb_id,55.86
has_valid_runtime,43.51
is_feature_length,37.12


In [11]:
# Semantic quality score
master["semantic_quality_score"] = 0.0

master["semantic_quality_score"] += master["has_usable_title"].astype(float) * 2.0
master["semantic_quality_score"] += (master["overview_word_count"].fillna(0) >= 10).astype(float) * 2.0
master["semantic_quality_score"] += (master["overview_word_count"].fillna(0) >= 25).astype(float) * 3.0
master["semantic_quality_score"] += (master["overview_word_count"].fillna(0) >= 50).astype(float) * 2.0
master["semantic_quality_score"] += master["has_genres"].astype(float) * 1.5
master["semantic_quality_score"] += master["has_director"].astype(float) * 1.0
master["semantic_quality_score"] += master["has_cast"].astype(float) * 1.0
master["semantic_quality_score"] += master["is_feature_length"].astype(float) * 2.0
master["semantic_quality_score"] += master["has_valid_year"].astype(float) * 1.5
master["semantic_quality_score"] -= master["is_tv_movie"].astype(float) * 2.0
master["semantic_quality_score"] -= master["is_future_release"].astype(float) * 3.0

# Documentation score
master["documentation_score"] = 0.0
master["documentation_score"] += master["has_imdb_id"].astype(float) * 1.0
master["documentation_score"] += (master["vote_count"].fillna(0) > 0).astype(float) * 1.0
master["documentation_score"] += (master["vote_count"].fillna(0) >= 10).astype(float) * 2.0
master["documentation_score"] += (master["vote_count"].fillna(0) >= 50).astype(float) * 2.0
master["documentation_score"] += (master["popularity"].fillna(0) >= 1).astype(float) * 1.0
master["documentation_score"] += (master["popularity"].fillna(0) >= 5).astype(float) * 2.0
master["documentation_score"] += master["has_audience_signal"].astype(float) * 1.0

# Outcome score
master["outcome_score"] = 0.0
master["outcome_score"] += master["revenue_clean"].notna().astype(float) * 2.0
master["outcome_score"] += master["budget_clean"].notna().astype(float) * 1.0
master["outcome_score"] += master["roi_clean"].notna().astype(float) * 1.0
master["outcome_score"] += (master["vote_count"].fillna(0) >= 10).astype(float) * 1.0
master["outcome_score"] += (master["vote_average"].fillna(0) > 0).astype(float) * 1.0

# Overall row quality
master["row_quality_score"] = (
    master["semantic_quality_score"] * 0.55 +
    master["documentation_score"] * 0.30 +
    master["outcome_score"] * 0.15
)

master["row_quality_score"] = master["row_quality_score"].round(3)

display(
    master[[
        "semantic_quality_score", "documentation_score", "outcome_score", "row_quality_score"
    ]].describe()
)

,semantic_quality_score,documentation_score,outcome_score,row_quality_score
count,1.192309e+06,1.192309e+06,1.192309e+06,1.192309e+06
mean,9.650478e+00,2.343923e+00,5.397988e-01,6.091910e+00
std,3.567504e+00,1.571301e+00,9.360996e-01,2.259975e+00
min,-1.000000e+00,0.000000e+00,0.000000e+00,-2.500000e-01
25%,6.500000e+00,1.000000e+00,0.000000e+00,4.175000e+00
50%,1.000000e+01,2.000000e+00,0.000000e+00,6.250000e+00
75%,1.300000e+01,3.000000e+00,1.000000e+00,7.750000e+00
max,1.500000e+01,1.000000e+01,6.000000e+00,1.215000e+01


In [12]:
dedup_log = []

work = master.copy()

# helper keys
work["title_key"] = normalize_title(work["title"])
work["title_year_key"] = np.where(
    work["title_key"].str.len() > 0 & work["release_year"].notna(),
    work["title_key"] + "||" + work["release_year"].fillna(-1).astype(int).astype(str),
    np.nan
)

# Step 1: exact id dedupe
before = len(work)
work = (
    work.sort_values(["id", "row_quality_score"], ascending=[True, False])
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)
dedup_log.append({
    "step": "dedupe_by_id",
    "rows_before": before,
    "rows_after": len(work),
    "rows_removed": before - len(work)
})

# Step 2: imdb_id dedupe for non-empty imdb ids
before = len(work)
with_imdb = work[work["imdb_id"].str.len() > 0].copy()
without_imdb = work[work["imdb_id"].str.len() == 0].copy()

with_imdb = (
    with_imdb.sort_values(["imdb_id", "row_quality_score"], ascending=[True, False])
    .drop_duplicates(subset=["imdb_id"], keep="first")
)

work = pd.concat([with_imdb, without_imdb], ignore_index=True)
dedup_log.append({
    "step": "dedupe_by_imdb_id",
    "rows_before": before,
    "rows_after": len(work),
    "rows_removed": before - len(work)
})

# Step 3: conservative title-year dedupe only for rows with missing imdb_id
before = len(work)
ty_subset = work[
    (work["imdb_id"].str.len() == 0) &
    (work["title_key"].str.len() > 0) &
    (work["release_year"].notna())
].copy()

rest_subset = work.drop(ty_subset.index).copy()

ty_subset = (
    ty_subset.sort_values(["title_year_key", "row_quality_score"], ascending=[True, False])
    .drop_duplicates(subset=["title_year_key"], keep="first")
)

work = pd.concat([rest_subset, ty_subset], ignore_index=True)
dedup_log.append({
    "step": "dedupe_by_title_year_missing_imdb_only",
    "rows_before": before,
    "rows_after": len(work),
    "rows_removed": before - len(work)
})

# final cleanup
work = work.drop(columns=["title_key", "title_year_key"], errors="ignore")
work = work.reset_index(drop=True)

dedup_log_df = pd.DataFrame(dedup_log)
display(dedup_log_df)

print("Final primary master shape:", work.shape)

,step,rows_before,rows_after,rows_removed
0,dedupe_by_id,1192309,1192309,0
1,dedupe_by_imdb_id,1192309,1192308,1
2,dedupe_by_title_year_missing_imdb_only,1192308,1184073,8235


Final primary master shape: (1184073, 49)


In [13]:
movies_per_year = (
    work["release_year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
movies_per_year.columns = ["release_year", "movie_count"]

audit_rows = []

def add_metric(name, value):
    audit_rows.append({"metric": name, "value": value})

add_metric("rows_final", len(work))
add_metric("cols_final", len(work.columns))
add_metric("year_min", int(work["release_year"].dropna().min()))
add_metric("year_max", int(work["release_year"].dropna().max()))
add_metric("pct_released_rows", round(work["is_released_row"].mean() * 100, 2))
add_metric("pct_future_release", round(work["is_future_release"].mean() * 100, 2))
add_metric("pct_feature_length", round(work["is_feature_length"].mean() * 100, 2))
add_metric("pct_usable_overview", round(work["has_usable_overview"].mean() * 100, 2))
add_metric("pct_has_genres", round(work["has_genres"].mean() * 100, 2))
add_metric("pct_has_cast", round(work["has_cast"].mean() * 100, 2))
add_metric("pct_has_director", round(work["has_director"].mean() * 100, 2))
add_metric("median_overview_words", float(work["overview_word_count"].median()))
add_metric("median_runtime", float(work["runtime"].median()))
add_metric("median_vote_count", float(work["vote_count"].median()))
add_metric("median_popularity", float(work["popularity"].median()))
add_metric("pct_vote_count_ge_10", round((work["vote_count"].fillna(0) >= 10).mean() * 100, 2))
add_metric("pct_vote_count_ge_50", round((work["vote_count"].fillna(0) >= 50).mean() * 100, 2))
add_metric("pct_revenue_clean", round(work["revenue_clean"].notna().mean() * 100, 2))
add_metric("pct_budget_clean", round(work["budget_clean"].notna().mean() * 100, 2))
add_metric("pct_roi_clean", round(work["roi_clean"].notna().mean() * 100, 2))

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
display(movies_per_year.tail(25))

,metric,value
0,rows_final,1184073.00
1,cols_final,49.00
2,year_min,1874.00
3,year_max,2031.00
4,pct_released_rows,98.16
5,pct_future_release,0.24
6,pct_feature_length,37.33
7,pct_usable_overview,77.96
8,pct_has_genres,73.17
9,pct_has_cast,68.34


,release_year,movie_count
122,2007,16711
123,2008,17468
124,2009,18440
125,2010,19215
126,2011,20779
127,2012,22674
128,2013,25048
129,2014,26841
130,2015,28129
131,2016,29475


In [14]:
print("Missing values (%) in imdb_master_primary:")
display(
    (work.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
    .head(25)
)

Missing values (%) in imdb_master_primary:


,0
movieId,100.00
roi_clean,98.51
roi_raw,98.51
revenue_clean,97.69
budget_clean,93.42
release_year,10.61
release_date,10.61
imdb_id,0.00
id,0.00
runtime,0.00


In [15]:
final_cols = [
    "id", "movieId", "imdb_id", "title", "overview", "keywords_clean",
    "cast_clean", "director_name", "genres_clean", "original_language",
    "release_date", "release_year", "budget", "revenue", "roi_raw",
    "runtime", "popularity", "vote_average", "vote_count",
    "budget_clean", "revenue_clean", "roi_clean",
    "overview_len", "overview_word_count",
    "text_overview_only", "text_semantic_enriched", "text_semantic_full",
    "status", "source_name",
    "is_released_row", "is_future_release", "has_valid_year",
    "has_usable_title", "has_usable_overview", "has_valid_runtime",
    "is_feature_length", "is_short_runtime", "has_genres", "has_cast",
    "has_director", "has_imdb_id", "has_finance_signal",
    "has_audience_signal", "is_tv_movie", "has_valid_language",
    "semantic_quality_score", "documentation_score",
    "outcome_score", "row_quality_score"
]

final_cols = [c for c in final_cols if c in work.columns]
work = work[final_cols].copy()

print("Final ordered primary master shape:", work.shape)
display(work.head(3))

Final ordered primary master shape: (1184073, 49)


,id,movieId,imdb_id,title,overview,keywords_clean,cast_clean,director_name,genres_clean,original_language,...,has_director,has_imdb_id,has_finance_signal,has_audience_signal,is_tv_movie,has_valid_language,semantic_quality_score,documentation_score,outcome_score,row_quality_score
0,16612,<NA>,tt0000001,Carmencita,The first woman to appear in front of an Ediso...,,Carmencita,,Documentary,xx,...,False,True,False,True,False,True,13.0,7.0,2.0,9.55
1,16613,<NA>,tt0000002,Clown and His Dogs,Short film of 300 individually painted images....,,,,Animation,fr,...,False,True,False,True,False,True,7.0,6.0,2.0,5.95
2,88013,<NA>,tt0000003,Poor Pierrot,"One night, Arlequin comes to see his lover Col...",,,,Comedy Animation,fr,...,False,True,False,True,False,True,10.0,7.0,2.0,7.90


In [16]:
work.to_csv(PRIMARY_OUT, index=False)
audit_df.to_csv(AUDIT_OUT, index=False)
dedup_log_df.to_csv(DEDUP_LOG_OUT, index=False)
movies_per_year.to_csv(YEAR_COUNT_OUT, index=False)

print("Saved:")
print("-", PRIMARY_OUT)
print("-", AUDIT_OUT)
print("-", DEDUP_LOG_OUT)
print("-", YEAR_COUNT_OUT)

Saved:
- /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary.csv
- /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary_audit.csv
- /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary_dedup_log.csv
- /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary_movies_per_year.csv


In [17]:
print("FINAL STATUS")
print("------------")
print("Primary IMDb master shape:", work.shape)
print("Saved primary file:", PRIMARY_OUT)

print("\nThis file is the new upstream source for Notebook 2.")
print("Next notebook should build:")
print("- df_semantic_analysis.csv")
print("- df_revenue_analysis.csv")
print("- df_roi_analysis.csv")

FINAL STATUS
------------
Primary IMDb master shape: (1184073, 49)
Saved primary file: /content/drive/MyDrive/thesis_movie_project/processed/imdb_master_primary.csv

This file is the new upstream source for Notebook 2.
Next notebook should build:
- df_semantic_analysis.csv
- df_revenue_analysis.csv
- df_roi_analysis.csv
